# Adam Optimizer From Scratch

**Topic:** Derivatives and Optimization Review &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `f(w)` and `grad_f(w)`: a 10-dimensional Rosenbrock-style valley function and its exact gradient (both provided, so no differentiation is needed).
- `w0`: the fixed, far-away starting point.
- The global minimum has `f(w*) = 0` at `w* = (1, 1, …, 1)`.

## Your task

Implement **momentum GD or Adam yourself** (no `scipy.optimize`, no torch optimizers) and run it from `w0` for at most 20,000 iterations.

Set `w_opt` to your final parameter vector.

Adam recap: `m ← β₁m + (1−β₁)g`, `v ← β₂v + (1−β₂)g²`, bias-correct both, then `w ← w − lr · m̂ / (√v̂ + ε)` with lr≈0.01, β₁=0.9, β₂=0.999.

## Evaluation

The evaluation cell checks that `f(w_opt) < 0.01` (plain GD at a conservative safe rate such as `1e-4` stalls far above this).


In [ ]:
import numpy as np

DIM = 10

def f(w):
    w = np.asarray(w, dtype=float)
    return float(np.sum(100.0 * (w[1:] - w[:-1] ** 2) ** 2 + (1 - w[:-1]) ** 2))

def grad_f(w):
    w = np.asarray(w, dtype=float)
    g = np.zeros_like(w)
    g[:-1] += -400.0 * w[:-1] * (w[1:] - w[:-1] ** 2) - 2.0 * (1 - w[:-1])
    g[1:] += 200.0 * (w[1:] - w[:-1] ** 2)
    return g

w0 = np.full(DIM, -1.5)
print(f"f(w0) = {f(w0):.1f}   (goal: < 0.01)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: how low did your optimizer get?
# =====================================================================
import numpy as np

if "w_opt" not in dir():
    raise NameError("Define `w_opt` in the solution cell before running this cell.")

_w = np.asarray(w_opt, dtype=float).ravel()
if _w.shape != (DIM,):
    raise ValueError(f"`w_opt` has shape {_w.shape}, expected ({DIM},).")
_val = f(_w)
print("=" * 52)
print(f"f(w_opt)          : {_val:.6f}   (target: < 0.01)")
print(f"distance to (1,…,1): {float(np.linalg.norm(_w - 1.0)):.4f}")
print("=" * 52)
print("PASSED: your optimizer works!" if _val < 0.01 else "NOT PASSED yet: try Adam, or more iterations.")


## Solution

Textbook Adam: first and second moment accumulators with bias correction. The per-coordinate step size lets it crawl along the curved Rosenbrock valley where plain gradient descent stalls.

```python
import numpy as np

w = np.asarray(w0, dtype=float).copy()
m = np.zeros_like(w)
v = np.zeros_like(w)
lr, b1, b2, eps = 0.01, 0.9, 0.999, 1e-8

for t in range(1, 20001):
    g = grad_f(w)
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    mh = m / (1 - b1 ** t)
    vh = v / (1 - b2 ** t)
    w -= lr * mh / (np.sqrt(vh) + eps)

w_opt = w
print(f"f(w_opt) = {f(w_opt):.6f}")
```
